# This notebook shows you how to generate your own DBOF


In [3]:
import matplotlib.pyplot as plt
import numpy as np
import datetime

# Initial set up
In the future we will want to update this to use docker or conda for end users

## These steps help you build the project on a local machine. 
If you are using nrp Jupyterhub, it is recommended you use the next block instead.

#### Build the project
- ```pip install .```

#### Install aws cli (optional)
You will want to install this if you want to manually see the data stored in the s3 bucket.

Example for installing on linux :
- ```sudo curl "https://awscli.amazonaws.com/awscli-exe-linux-x86_64.zip" -o "awscliv2.zip"```
- ```sudo unzip awscliv2.zip```
- ```sudo ./aws/install```

## Running in NRP Jupyterhub

This is for running this notebook on nrp Jupyterhub. 

This block simply builds the project and installs dependencies not already present on nrp jupyterhub.
You can safely ignore pip warnings

For serious projects, users should use conda or docker but this notebook is meant to be very simple and user friendly.

In [6]:
#Set True if running on NRP Jupyterhub
RUNNING_ON_NRP = False


if (RUNNING_ON_NRP):
    %pip install -e ../. --no-deps

    %pip install xgcm
    %pip install zarr
    %pip install boto3
    %pip install ujson
    %pip install scikit-fmm
    %pip install aiobotocore

# NOTE IF on NRP Jupyterhub, you will likely need to restart the kernel after running this block

Obtaining file:///home/jovyan/git/llc4320-native-grid-preprocessing
  Installing build dependencies ... one
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for dbof-in-native-grid (pyproject.toml) ... done
  Created wheel for dbof-in-native-grid: filename=dbof_in_native_grid-0.0.dev0-0.editable-py3-none-any.whl size=3209 sha256=ad8d3ab3e8794e40d26cf3685499f96bc23d7e02339ebb9cc4d80485400f36d3
  Stored in directory: /tmp/pip-ephem-wheel-cache-npfuy8ze/wheels/1a/4c/86/ce01074c4f718570f5d3513e6430e9f97f383cf210efd719bf
Successfully built dbof-in-native-grid
  Attempting uninstall: dbof-in-native-grid
    Found existing installation: dbof-in-native-grid 0.0.dev0
    Uninstalling dbof-in-native-grid-0.0.dev0:
      Successfully uninstalled dbof-in-native-grid-0.0.dev0

[notice] A new release of pip is available: 25.2 -> 25.3
[notice] To update, run: pip i

## Set AWS credentials
If you are accessing or writing data to S3, you must set credentials.
NOTE : S3 is all that is supported currently.
This typically corresponds to an NRP S3 bucket.

windows:

- ```$env:AWS_ACCESS_KEY_ID="..."```
- ```$env:AWS_SECRET_ACCESS_KEY="..."```

unix:

- ```export AWS_ACCESS_KEY_ID=...```
- ```export AWS_SECRET_ACCESS_KEY=...```

# Dataset generation config (quick reference)

This job is fully controlled by a YAML config file. The config defines **what time range is scanned**, **how often snapshots are taken**, and **how many spatial patches are sampled per snapshot**.

### Temporal sampling
- `data.timestep_hours`
  Total time window (in hours) to scan starting from `start_record`.
  Example: `8760` = one LLC model year (336 days).

- `data.sampling_step`
  Spacing **in hours** between snapshots within the window.
  Example: `2190` with `timestep_hours=8760` → 4 evenly spaced snapshots.

- `data.start_record`
  First valid wind/forcing record (default: `1180`). You probably don't want to change this.

Each snapshot corresponds to a **single instantaneous model timestep** (not an average).

### Spatial sampling
- `sampling.sample_points_per_snapshot`
  Number of cutouts sampled per snapshot.

- `sampling.bias_to_high_gradients`
  Exponential bias favoring high-gradient regions when sampling.

### Patch geometry
- `output.target_km_res`
  Physical patch size in km. Default 150.

- `output.down_sample_res`
  Pixel resolution of the extracted cutouts.
  Must be small enough to safely resolve `target_km_res` on the LLC grid.
  Default 64

### Output / logging
- `output.bucket`, `output.folder`
  S3 location for dataset output.

- `run.run_id`
  Unique identifier for this run (used for logs and output paths).
  If the run_id you use exists on the write location (s3 bucket), your data will be appended to the existing data from previous run(s).

- `run.log_dir`
  Local directory where logs are written.

### Invariants (not configurable)
Grid topology, model cadence (144 timesteps/hour), and dataset offsets are fixed
LLC4320 constants and are enforced in code.

### Examples
See existing configs in configs/ for examples

# A note about logs
The run logs will be stored locally on your machine in the directory you specify.
- ```log_dir/run_id/```
However under the current logic, if you attempt to run the script and the specified log output path already exists, the script will fail to run. This is by design. The reasons are as follows.
- run_id is also used for the zarr dataset path. You likely do not want to send two different runs to the same dataset.
- You will likely not want to override your previous run logs.

If you want to override this logic, simply delete the existing log output path from your local machine. You can also override the run_id with a cli argument ```--run_id```

## Example run 1
- 1 year of data
- 4 time snapshots evenly spaced through the year
- 150 cutouts per timestamp

If you are running this on your laptop or pc expect it to take a while.

In [3]:
run_id = f"year_4x150_{datetime.datetime.now(datetime.UTC).strftime('%Y%m%d_%H%M%S')}"
print(f"Your run id is: {run_id}")
print("Make sure and use the same run_d later when accessing your data")

Your run id is: year_4x150_20260129_153218
Make sure and use the same run_d later when accessing your data


In [4]:
# note: that --run_id can overwrite the config file
# note: All Dask logs are warnings. Don't be alarmed.
!generate-llc-dataset --config ../configs/example_1year_4_snapshot.yaml --run_id {run_id}

2026-01-29 15:32:57,711 | INFO | Arguments parsed successfully. Logging set up. Running script.
/opt/conda/lib/python3.12/site-packages/distributed/node.py:187: UserWarning: Port 8787 is already in use.
Perhaps you already have a cluster running?
Hosting the HTTP server on port 44457 instead
  warnings.warn(
2026-01-29 15:33:10,418 | INFO | Dask Client <Client: 'tcp://127.0.0.1:39085' processes=10 threads=80, memory=128.00 GiB>
2026-01-29 15:33:10,418 | INFO | Processing: [ 180288  495648  811008 1126368] time snapshots
2026-01-29 15:33:10,758 | INFO | Found credentials in shared credentials file: ~/.aws/credentials
2026-01-29 15:33:12,076 | INFO | Zarr dataset_creation created.
2026-01-29 15:33:12,076 | INFO | Fetching grid file
2026-01-29 15:33:17,859 | INFO | Calculating land and face masks
^C


## Example run 2 - very small
- 2 weeks of data
- 1 snapshot per week
- 10 cutouts per timestamp

If you are running this on your laptop or pc expect it to take a while.

In [7]:
run_id = f"small_test{datetime.datetime.now(datetime.UTC).strftime('%Y%m%d_%H%M%S')}"
print(f"Your run id is: {run_id}")
print("Make sure and use the same run_d later when accessing your data")

Your run id is: small_test20260129_162702
Make sure and use the same run_d later when accessing your data


In [8]:
!generate-llc-dataset --config ../configs/test.yaml --run_id {run_id}

2026-01-29 16:27:16,668 | INFO | Arguments parsed successfully. Logging set up. Running script.
/opt/conda/lib/python3.12/site-packages/distributed/node.py:187: UserWarning: Port 8787 is already in use.
Perhaps you already have a cluster running?
Hosting the HTTP server on port 43651 instead
  warnings.warn(
2026-01-29 16:27:32,269 | INFO | Dask Client <Client: 'tcp://127.0.0.1:37139' processes=10 threads=80, memory=128.00 GiB>
2026-01-29 16:27:32,269 | INFO | Processing: [180288 204480] time snapshots
2026-01-29 16:27:32,622 | INFO | Found credentials in shared credentials file: ~/.aws/credentials
/opt/conda/lib/python3.12/site-packages/zarr/core/dtype/npy/bytes.py:386: UnstableSpecificationWarning: The data type (NullTerminatedBytes(length=32)) does not have a Zarr V3 specification. That means that the representation of arrays saved with this data type may change without warning in a future version of Zarr Python. Arrays stored with this data type may be unreadable by other Zarr libr

In [ ]:
# !pip install s3fs
# aws --endpoint https://s3-west.nrp-nautilus.io s3 ls s3://llc/native_grid_dbof_training_data/script_test_00/ --human-readable
#
# aws --endpoint https://s3-west.nrp-nautilus.io s3 rm s3://llc/native_grid_dbof_training_data/script_test_00/ --recursive --dryrun